# 01 - Data Collection & Generation

## CASEFILE: AI-Powered Missing Person Investigation System

### Project Context & Objectives
In missing person investigations, time is critical. Traditional search operations often rely on manual search grids, anecdotal witness statements, and retrospective phone ping triangulation. **CASEFILE** applies modern spatial intelligence and machine learning to GPS trajectory streams to rapidly model habitual mobility patterns, detect behavioral anomalies, and prioritize search zones.

This notebook documents the **Data Collection and Generation Phase** of the CASEFILE pipeline.

### Synthetic GPS Data Generation Approach (Beijing Area)
Real-world continuous GPS tracking data for human subjects is heavily restricted due to privacy, security, and ethical considerations. To facilitate research and algorithm benchmarking without compromising personal privacy, CASEFILE utilizes a high-fidelity synthetic trajectory generator calibrated to the **Beijing Metropolitan Area**.

#### Simulation Methodology (`src/data_collection.py`):
1. **Geographic Bounding Box**:
   - Latitude: $39.85^\circ \text{N}$ to $40.05^\circ \text{N}$
   - Longitude: $116.20^\circ \text{E}$ to $116.55^\circ \text{E}$
   - Altitude: Centered around $50\text{m}$ with Gaussian deviation ($\sigma = 5\text{m}$)
2. **Behavioral Archetypes & Life Anchors**:
   - Each synthetic user is assigned a synthetic **Home** anchor and a **Work** anchor located $2 - 8\text{ km}$ away.
   - Distinct sets of **Frequent POIs** (e.g., lunch spots, transit stops) and **Occasional POIs** (e.g., parks, shopping markets).
3. **Temporal Dynamics**:
   - Weekday routines model morning commutes (07:30 - 08:30), workplace stays, midday lunch excursions, afternoon work sessions, and evening commutes.
   - Weekend routines introduce spontaneous recreational trips and varied departure times.
4. **Physical Kinematics & Noise Injection**:
   - Intermediate trajectory waypoints are generated at 15-second sampling intervals during motion and 60-second intervals during stationary dwell periods.
   - Realistic Gaussian jitter ($\sigma \approx 10-15\text{ meters}$) is superimposed to simulate consumer GPS multipath inaccuracies and satellite signal drift.

---
### Ethical Disclaimer & Academic Integrity
> **Notice on Synthetic Data & Privacy Compliance**:
> Trajectory and location data represent highly sensitive Personally Identifiable Information (PII). Uncontrolled disclosure of real geolocation traces can expose residences, daily habits, medical appointments, and intimate relationships.
> 
> In strict adherence to ethical research standards, data protection regulations (such as GDPR and PIPL), and privacy-preserving AI principles:
> 1. All mobility traces analyzed in this demonstration were generated synthetically using mathematical trajectory modeling.
> 2. No real individuals, vehicles, or devices were tracked or surveilled.
> 3. The synthetic Point of Interest (POI) coordinates are procedurally distributed across Beijing municipal coordinates for spatial simulation purposes.
---

In [ ]:
import os
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Configure aesthetic visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

print("Environment initialized and sys.path configured.")

### 1. Inspecting the Synthetic Data Generator (`src/data_collection.py`)
Let us verify the generator parameters defined in `src/data_collection.py`.

In [ ]:
import sys
sys.path.insert(0, '..')
import src.data_collection as dc

print("--- Synthetic Generator Configuration ---")
print(f"Beijing Bounding Box Latitude:  [{dc.BEIJING_LAT_MIN}, {dc.BEIJING_LAT_MAX}]")
print(f"Beijing Bounding Box Longitude: [{dc.BEIJING_LON_MIN}, {dc.BEIJING_LON_MAX}]")
print(f"Number of Simulated Users:      {dc.NUM_USERS}")
print(f"Simulation Start Date:          {dc.START_DATE.strftime('%Y-%m-%d')}")
print(f"Active Days per User:           {dc.DAYS_PER_USER_MIN} to {dc.DAYS_PER_USER_MAX} days")

### 2. Loading Raw Datasets
We load the raw synthetic GPS trajectories (`data/raw/gps_trajectories.csv`) and the Beijing Points of Interest (`data/raw/beijing_pois.csv`).

In [ ]:
import sys
sys.path.insert(0, '..')

raw_traj_path = os.path.join('..', 'data', 'raw', 'gps_trajectories.csv')
raw_pois_path = os.path.join('..', 'data', 'raw', 'beijing_pois.csv')

df_raw = pd.read_csv(raw_traj_path)
df_pois = pd.read_csv(raw_pois_path)

print(f"Raw GPS Trajectories count: {len(df_raw):,} records")
print(f"Raw POIs count:             {len(df_pois):,} landmarks")
print("\n--- Sample GPS Trajectory Records ---")
display(df_raw.head())
print("\n--- Sample Points of Interest (POIs) ---")
display(df_pois.head())

### 3. Summary Statistics: Trajectory Counts, Date Ranges & Spatial Bounds
Let us analyze the temporal extent, spatial bounding boxes, and volume of records per user.

In [ ]:
import sys
sys.path.insert(0, '..')

df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])

user_summary = df_raw.groupby('user_id').agg(
    total_points=('timestamp', 'count'),
    first_seen=('timestamp', 'min'),
    last_seen=('timestamp', 'max'),
    min_lat=('latitude', 'min'),
    max_lat=('latitude', 'max'),
    min_lon=('longitude', 'min'),
    max_lon=('longitude', 'max'),
    avg_altitude_m=('altitude', 'mean')
).reset_index()

user_summary['days_covered'] = (user_summary['last_seen'] - user_summary['first_seen']).dt.days + 1
user_summary['points_per_day'] = (user_summary['total_points'] / user_summary['days_covered']).round(1)

print("User Trajectory Metrics Summary:")
display(user_summary)

### 4. Overall Spatial Bounding Box Verification
Checking whether coordinates respect the expected geographic boundaries of Beijing.

In [ ]:
import sys
sys.path.insert(0, '..')

bbox_stats = pd.DataFrame({
    'Dimension': ['Latitude (°N)', 'Longitude (°E)', 'Altitude (m)'],
    'Min Observed': [df_raw['latitude'].min(), df_raw['longitude'].min(), df_raw['altitude'].min()],
    'Max Observed': [df_raw['latitude'].max(), df_raw['longitude'].max(), df_raw['altitude'].max()],
    'Mean Observed': [df_raw['latitude'].mean(), df_raw['longitude'].mean(), df_raw['altitude'].mean()],
    'Std Dev': [df_raw['latitude'].std(), df_raw['longitude'].std(), df_raw['altitude'].std()]
})

print("Dataset Spatial Boundary Diagnostics:")
display(bbox_stats)

### 5. Visualizations: Spatial Distribution & POI Breakdown
We visualize:
1. **GPS Point Distribution on a Scatter Plot**: Downsampled overlay showing individual mobility corridors across Beijing with landmark POIs.
2. **POI Category Distribution**: A breakdown of municipal facilities and interest points.

In [ ]:
import sys
sys.path.insert(0, '..')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 1. Spatial Scatter Plot
sample_df = df_raw.sample(n=35000, random_state=42)
scatter = axes[0].scatter(
    sample_df['longitude'], sample_df['latitude'],
    c=sample_df['user_id'], cmap='tab10',
    alpha=0.25, s=2, label='GPS Trajectory Points (Sampled)'
)
axes[0].scatter(
    df_pois['longitude'], df_pois['latitude'],
    color='red', marker='X', s=70, edgecolors='black', linewidths=0.8,
    label='Points of Interest (POIs)'
)
axes[0].set_title('GPS Point Distribution & POIs (Beijing Area)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Longitude (°E)', fontsize=11)
axes[0].set_ylabel('Latitude (°N)', fontsize=11)
axes[0].legend(loc='upper right')
axes[0].grid(True, linestyle='--', alpha=0.6)

# 2. POI Category Distribution
poi_counts = df_pois['category'].value_counts()
sns.barplot(x=poi_counts.values, y=poi_counts.index, ax=axes[1], palette='viridis')
axes[1].set_title('POI Category Distribution in Beijing Area', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of POIs', fontsize=11)
axes[1].set_ylabel('Category', fontsize=11)

for idx, val in enumerate(poi_counts.values):
    axes[1].text(val + 0.2, idx, str(val), va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

### 6. Temporal Volume & User Balance
Visualizing the volume of data generated per user over time.

In [ ]:
import sys
sys.path.insert(0, '..')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Total points per user
user_counts = df_raw['user_id'].value_counts().sort_index()
sns.barplot(x=user_counts.index, y=user_counts.values, ax=axes[0], palette='crest')
axes[0].set_title('Total GPS Points per User', fontsize=13, fontweight='bold')
axes[0].set_xlabel('User ID', fontsize=11)
axes[0].set_ylabel('Point Count', fontsize=11)
for i, v in enumerate(user_counts.values):
    axes[0].text(i, v + 1500, f"{v//1000}k", ha='center', fontsize=9)

# Date distribution of points
daily_counts = df_raw.set_index('timestamp').resample('D').size()
axes[1].plot(daily_counts.index, daily_counts.values, color='#2b5c8f', lw=1.8)
axes[1].set_title('Daily Trajectory Recording Volume Across Study Period', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=11)
axes[1].set_ylabel('Total Points / Day', fontsize=11)
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### Summary of Phase 1
- **Dataset Scale**: Loaded **902,052** raw GPS records from `data/raw/gps_trajectories.csv` covering 10 users across 30–60 days.
- **Geographic Validity**: Verified coordinates adhere to Beijing municipal coordinates ($39.85^\circ\text{N} - 40.05^\circ\text{N}$, $116.20^\circ\text{E} - 116.55^\circ\text{E}$).
- **POIs**: Integrated **52** categorized urban points of interest providing semantic context for destinations.
- **Next Phase**: Proceed to **`02_data_preprocessing.ipynb`** to clean duplicate observations, filter sensor noise, and handle kinematic speed outliers.